# UIT DSC 2026 LegalIR
Attach the competition-data and cached-artifacts datasets, then set the paths in `configs/kaggle_t4x2.yaml` if needed.

In [ ]:
# Set this to the attached competition-data dataset.
from pathlib import Path
DATASET_DIR = Path('/kaggle/input/REPLACE_WITH_DATASET_SLUG')

In [ ]:
!git clone https://github.com/REPLACE_WITH_YOUR_ORG/UIT-LegalIR.git
%cd UIT-LegalIR
for item in ('selected-contexts', 'train.json', 'public-official.json'):
    target = Path(item)
    if not target.exists():
        target.symlink_to(DATASET_DIR / item)
!pip install -q -e . -r requirements.txt

In [ ]:
import os, subprocess, sys
base = [sys.executable, '-m', 'legalir']
def run(*args, gpu=None):
    env = dict(os.environ)
    if gpu is not None: env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    return subprocess.run(base + list(args), check=True, env=env)

run('prepare', '--config', 'configs/kaggle_t4x2.yaml', '--resume')
run('audit', '--config', 'configs/kaggle_t4x2.yaml')
run('index', '--config', 'configs/kaggle_t4x2.yaml', '--lexical-only', '--resume')
p0 = subprocess.Popen(base + ['index', '--config', 'configs/kaggle_t4x2.yaml', '--model', 'vietlegal_e5', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
p1 = subprocess.Popen(base + ['index', '--config', 'configs/kaggle_t4x2.yaml', '--model', 'vietnamese_embedding', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '1'})
p0.wait(); p1.wait()
assert p0.returncode == p1.returncode == 0
run('index', '--config', 'configs/kaggle_t4x2.yaml', '--model', 'nemotron', '--resume', gpu=0)

In [ ]:
run('tune', '--config', 'configs/kaggle_t4x2.yaml', '--resume')
p0 = subprocess.Popen(base + ['rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'train', '--fold', '0', '--engine', 'jina', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
p1 = subprocess.Popen(base + ['rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'train', '--fold', '0', '--engine', 'vietnamese_reranker', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '1'})
p0.wait(); p1.wait()
assert p0.returncode == p1.returncode == 0
run('rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'train', '--fold', '0', '--resume')
run('tune', '--config', 'configs/kaggle_t4x2.yaml', '--final', '--fold', '0', '--resume')
run('retrieve', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'public', '--resume')
p0 = subprocess.Popen(base + ['rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'public', '--engine', 'jina', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
p1 = subprocess.Popen(base + ['rerank', '--config', 'configs/kaggle_t4x2.yaml', '--split', 'public', '--engine', 'vietnamese_reranker', '--resume'], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '1'})
p0.wait(); p1.wait()
assert p0.returncode == p1.returncode == 0
run('predict', '--config', 'configs/kaggle_t4x2.yaml', '--output', 'submission.json', '--resume')
subprocess.run(['zip', '-j', 'submission.zip', 'submission.json'], check=True)